In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ระบุ option

<details>
<summary><b>เวอร์ชันของแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาขึ้นโดยใช้ requirement ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
คุณสามารถใช้ option เพื่อปรับแต่ง Estimator และ Sampler primitives ส่วนนี้เน้นวิธีระบุ option ของ Qiskit Runtime primitive แม้ว่า interface ของเมธอด `run()` ของ primitives จะเหมือนกันในทุก implementation แต่ option ของแต่ละตัวนั้นต่างกัน ดูข้อมูลเพิ่มเติมเกี่ยวกับ option ของ [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) และ [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) ได้ใน API reference ที่เกี่ยวข้อง

หมายเหตุเกี่ยวกับการระบุ option ใน primitives:

- `SamplerV2` และ `EstimatorV2` มี option class แยกกัน คุณสามารถดู option ที่มีอยู่และอัปเดตค่า option ได้ระหว่างหรือหลังการ initialize primitive
- ใช้เมธอด `update()` เพื่อใช้การเปลี่ยนแปลงกับ attribute `options`
- ถ้าคุณไม่ระบุค่าสำหรับ option ใด มันจะได้รับค่าพิเศษ `Unset` และจะใช้ค่า default ของเซิร์ฟเวอร์
- attribute `options` เป็น Python type `dataclass` คุณสามารถใช้เมธอด `asdict` ในตัวเพื่อแปลงเป็น dictionary ได้

<span id="pass-options"></span>
## ตั้งค่า option ของ primitive
คุณสามารถตั้งค่า option ได้ตอน initialize primitive ตอนหลัง initialize primitive หรือใน เมธอด `run()` ดูส่วน [กฎลำดับความสำคัญ](runtime-options-overview#options-precedence) เพื่อทำความเข้าใจว่าเกิดอะไรขึ้นเมื่อ option เดียวกันถูกระบุในหลายที่

### การ initialize Primitive
คุณสามารถส่ง instance ของ option class หรือ dictionary ตอน initialize primitive ซึ่งจะทำสำเนา option เหล่านั้น ดังนั้นการเปลี่ยน dictionary หรือ option instance ต้นฉบับจะไม่ส่งผลกับ option ที่ primitive เป็นเจ้าของ

#### Option class
เมื่อสร้าง instance ของคลาส `EstimatorV2` หรือ `SamplerV2` คุณสามารถส่ง instance ของ option class เข้าไปได้ option เหล่านั้นจะถูกใช้เมื่อคุณใช้ `run()` เพื่อทำการคำนวณ ระบุ option ในรูปแบบนี้: `options.option.sub-option.sub-sub-option = choice` เช่น `options.dynamical_decoupling.enable = True`

ตัวอย่าง:

`SamplerV2` และ `EstimatorV2` มี option class แยกกัน ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) และ [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options))

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary
คุณสามารถระบุ option เป็น dictionary ตอน initialize primitive ได้

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### อัปเดต option หลัง initialization
คุณสามารถระบุ option ในรูปแบบนี้: `primitive.options.option.sub-option.sub-sub-option = choice` เพื่อใช้ประโยชน์จาก auto-complete หรือใช้เมธอด `update()` เพื่ออัปเดตหลายค่าพร้อมกัน

คลาส option ของ `SamplerV2` และ `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) และ [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) ไม่จำเป็นต้อง instantiate ถ้าคุณตั้งค่า option หลังจาก initialize primitive แล้ว

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### เมธอด Run()
ค่าที่คุณส่งให้ `run()` ได้มีเฉพาะที่กำหนดไว้ใน interface เท่านั้น กล่าวคือ `shots` สำหรับ Sampler และ `precision` สำหรับ Estimator การระบุค่าเหล่านี้จะ overwrite ค่าที่ตั้งไว้สำหรับ `default_shots` หรือ `default_precision` ในการ run ปัจจุบัน

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### กรณีพิเศษ
#### Resilience level (เฉพาะ Estimator)
Resilience level ไม่ใช่ option ที่ส่งผลโดยตรงต่อการ query primitive แต่เป็นการระบุชุด option ที่คัดสรรไว้แล้วเป็นฐาน โดยทั่วไป level 0 จะปิดการลดความผิดพลาดทั้งหมด, level 1 จะเปิด option สำหรับ measurement error mitigation และ level 2 จะเปิด option สำหรับ gate และ measurement error mitigation

option ที่คุณระบุเพิ่มเติมนอกเหนือจาก resilience level จะถูกใช้เพิ่มเติมบน option พื้นฐานที่กำหนดโดย resilience level ดังนั้นในหลักการ คุณสามารถตั้ง resilience level เป็น 1 แต่ปิด measurement mitigation ได้ แม้ว่าจะไม่แนะนำ

ในตัวอย่างต่อไปนี้ การตั้ง resilience level เป็น 0 จะปิด `zne_mitigation` ในตอนแรก แต่ `estimator.options.resilience.zne_mitigation = True` จะ override การตั้งค่าที่เกี่ยวข้องจาก `estimator.options.resilience_level = 0`

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Shots (เฉพาะ Sampler)
เมธอด `SamplerV2.run` รับอาร์กิวเมนต์สองตัว: รายการของ PUB แต่ละตัวสามารถระบุค่า shots เฉพาะ PUB ได้ และอาร์กิวเมนต์คีย์เวิร์ด shots ค่า shots เหล่านี้เป็นส่วนหนึ่งของ interface การรัน Sampler และเป็นอิสระจาก option ของ Runtime Sampler ค่าเหล่านี้มีลำดับความสำคัญเหนือกว่าค่าที่ระบุเป็น option เพื่อให้สอดคล้องกับ Sampler abstraction

อย่างไรก็ตาม ถ้า `shots` ไม่ได้ถูกระบุโดย PUB ใดเลยหรือในอาร์กิวเมนต์คีย์เวิร์ดของ run (หรือทั้งหมดเป็น `None`) ค่า shots จาก option จะถูกใช้ โดยเฉพาะ `default_shots`

สรุปคือ นี่คือลำดับความสำคัญสำหรับการระบุ shots ใน Sampler สำหรับแต่ละ PUB:

1. ถ้า PUB ระบุ shots ให้ใช้ค่านั้น
2. ถ้าอาร์กิวเมนต์คีย์เวิร์ด `shots` ถูกระบุใน `run` ให้ใช้ค่านั้น
3. ถ้า `num_randomizations` และ `shots_per_randomization` ถูกระบุเป็น option ของ `twirling` ค่า shots คือผลคูณของค่าทั้งสอง
3. ถ้า `sampler.options.default_shots` ถูกระบุ ให้ใช้ค่านั้น

ดังนั้น ถ้า shots ถูกระบุในทุกที่ที่เป็นไปได้ ตัวที่มีลำดับความสำคัญสูงสุด (shots ที่ระบุใน PUB) จะถูกใช้

#### Precision (เฉพาะ Estimator)
Precision มีลักษณะคล้ายกับ shots ที่อธิบายในส่วนก่อนหน้า ต่างกันที่ option ของ Estimator มีทั้ง `default_shots` และ `default_precision` นอกจากนี้ เนื่องจาก gate-twirling ถูกเปิดใช้งานโดย default ผลคูณของ `num_randomizations` และ `shots_per_randomization` จะมีลำดับความสำคัญเหนือ option ทั้งสองนั้น

โดยเฉพาะสำหรับแต่ละ Estimator PUB:

1. ถ้า PUB ระบุ precision ให้ใช้ค่านั้น
2. ถ้าอาร์กิวเมนต์คีย์เวิร์ด precision ถูกระบุใน `run` ให้ใช้ค่านั้น
2. ถ้า `num_randomizations` และ `shots_per_randomization` ถูกระบุเป็น [option ของ `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (เปิดใช้งานโดย default) ให้ใช้ผลคูณของทั้งสองเพื่อควบคุมปริมาณข้อมูล
3. ถ้า `estimator.options.default_shots` ถูกระบุ ให้ใช้ค่านั้นเพื่อควบคุมปริมาณข้อมูล
4. ถ้า `estimator.options.default_precision` ถูกระบุ ให้ใช้ค่านั้น

ตัวอย่างเช่น ถ้า precision ถูกระบุทั้งสี่ที่ ตัวที่มีลำดับความสำคัญสูงสุด (precision ที่ระบุใน PUB) จะถูกใช้

> **Note:** Precision สัมพันธ์แบบผกผันกับการใช้งาน กล่าวคือ ยิ่ง precision ต่ำยิ่งใช้เวลา QPU มากขึ้น

## Option ที่ใช้บ่อย
มี option ให้ใช้มากมาย แต่ต่อไปนี้คือที่ใช้บ่อยที่สุด:

<span id="shots"></span>
### Shots
สำหรับบาง algorithm การกำหนดจำนวน shots ที่แน่นอนเป็นส่วนสำคัญของกระบวนการ Shots (หรือ precision) สามารถระบุได้หลายที่ โดยมีลำดับความสำคัญดังนี้:

สำหรับ Sampler PUB ทุกตัว:

1. ค่า shots เป็นจำนวนเต็มที่อยู่ใน PUB
2. ค่า `run(...,shots=val)`
3. ค่า `options.default_shots`

สำหรับ Estimator PUB ทุกตัว:

1. ค่า precision เป็น float ที่อยู่ใน PUB
2. ค่า `run(...,precision=val)`
3. ค่า `options.default_shots`
4. ค่า `options.default_precision`

ตัวอย่าง:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### เวลาดำเนินการสูงสุด
เวลาดำเนินการสูงสุด (`max_execution_time`) จำกัดระยะเวลาที่ job สามารถรันได้ ถ้า job เกินขีดจำกัดนี้จะถูกยกเลิกโดยบังคับ ค่านี้ใช้กับ job เดียว ไม่ว่าจะรันใน job, session หรือ batch mode

ค่าถูกตั้งเป็นวินาทีโดยอิงจาก quantum time (ไม่ใช่ wall clock time) ซึ่งคือระยะเวลาที่ QPU ถูกจัดสรรเพื่อประมวลผล job ของคุณ ค่านี้จะถูกละเว้นเมื่อใช้ local testing mode เนื่องจาก mode นั้นไม่ใช้ quantum time